# 10 · Embedding 基础

> 文本 → 向量 → 相似度。这一课把“语义如何变成坐标、坐标距离如何近似语义”讲透。

**本文件覆盖知识点**：Dense Embedding / Vector Representation / Semantic Similarity / Cosine Similarity / Dot Product / Euclidean Distance

In [ ]:
# ===== 本课共用：真调 LLM 做「说明 / 演示」的小助手 =====
# 凡某个知识点能靠“真调一次大模型”当场讲清 / 演示的，下面的 cell 都用 _llm_live()
# 真调 qwen-plus 并打印模型输出作为说明；只有在项目根 .env 配了 DASHSCOPE_API_KEY 时才真调，
# 没配置就打印一段固定的演示样例，保证整个 notebook 不联网也能完整读下来。
from dotenv import load_dotenv; load_dotenv()
import os
from dashscope import Generation

_KEY = os.getenv('DASHSCOPE_API_KEY', '').strip()
_HAS_KEY = bool(_KEY) and '你的' not in _KEY

def _llm_live(prompt, fallback, system='你是资深 RAG 讲师，回答精炼、结构清晰、尽量结合例子。', temperature=0.3, model='qwen-plus'):
    """真调一次 qwen-plus 并打印结果；无 Key 时打印 fallback 作为演示样例。返回模型文本或 None。"""
    if not _HAS_KEY:
        print('未在 .env 配置 DASHSCOPE_API_KEY，跳过实时调用。以下是固定演示样例（配置后自动变为实时输出）：')
        print(fallback)
        return None
    msgs = [{'role': 'system', 'content': system}, {'role': 'user', 'content': prompt}]
    try:
        r = Generation.call(model=model, messages=msgs, temperature=temperature, result_format='message', api_key=_KEY)
        if r.status_code == 200:
            text = r.output.choices[0].message.content
            print('—— 模型实时输出 ——')
            print(text)
            return text
        print('调用失败：', getattr(r, 'code', ''), getattr(r, 'message', ''))
    except Exception as e:
        print('调用异常：', e)
    print('fallback：')
    print(fallback)
    return None


## 1. Dense Embedding：把语义压成坐标

Embedding 模型把任意文本映射成固定维度的稠密向量（每维都有值，故称 dense）。关键性质：

> 语义相近 → 向量方向相近 → 相似度分数高

```text
"客服机器人怎么收费"  →  [0.02, 0.87, -0.11, ...]   (1024维)
"价格套餐有哪些档位"  →  [0.03, 0.85, -0.09, ...]   ← 接近上面的向量
"今天吃了牛肉面"      →  [-0.5, 0.1, 0.6, ...]      ← 离得很远
```

In [ ]:
# API Key 通过 .env 配置（本项目统一方式）
from dotenv import load_dotenv; load_dotenv()
import os, numpy as np
API_KEY = os.getenv('DASHSCOPE_API_KEY', '')

def embed(texts, model='text-embedding-v3'):
    """百炼文本向量化：返回 L2 归一化后的 float32 矩阵 (n, dim)"""
    from dashscope import TextEmbedding
    if isinstance(texts, str):
        texts = [texts]
    r = TextEmbedding.call(model=model, input=texts, api_key=API_KEY)
    if r.status_code != 200:
        raise RuntimeError(f'Embedding 失败: {r.status_code} {r.message}')
    a = np.array([e['embedding'] for e in r.output['embeddings']], dtype='float32')
    return a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-10)   # 归一化

if not API_KEY or '你的' in API_KEY:
    print('⚠ 请先在 .env 中配置 DASHSCOPE_API_KEY（安装 dashscope: pip install dashscope）')
else:
    texts = ['客服机器人怎么收费', '价格套餐有哪些档位', '今天吃了牛肉面']
    v = embed(texts)
    print('向量矩阵形状:', v.shape, '（n=文本数, dim=维度）')
    print('每条向量模长≈1（已归一化）:', np.round(np.linalg.norm(v[0]), 3))

In [ ]:
# 知识点·真调说明：Dense 向量 · 稠密与稀疏 —— 让 LLM 讲清“稠密”为何能表语义、与 one-hot/词袋差在哪
# 上一格是真向量演示（近义句方向接近）；这里补上“为什么必须是稠密坐标”的概念。
_llm_live(
    prompt="""用大白话加一个中文例子讲清楚：为什么说 embedding 向量是“稠密”的？它与 one-hot 独热编码、词袋这种“稀疏”表示的本质差别是什么？这种差别如何让“意思相近但用词不同”的两句话在向量上更接近？""",
    system='你是资深 RAG / 表示学习讲师，回答控制在 6 句以内、结构清晰，必须带一个中文近义句例子。',
    fallback="""one-hot/词袋的维度是“词”，一句话只点亮它含有的那几个词，其余维全是 0，所以稀疏，且“价格”和“收费”两个不同词永远无法在向量上靠近。
embedding 把整句语义压缩进一个每维都有值的固定长度稠密向量（如 1024 维），相近语义会落在相近方向上。
例：“客服机器人怎么收费”和“价格套餐有哪些档位”用词几乎不重叠，但在稠密空间里方向很接近——靠语义而非字面。""",
    temperature=0.2,
)
print('→ 稠密向量正是“语义相近 → 方向相近”的载体，也是 dense retrieval 能命中近义问法的前提。')

## 2. 三种相似度度量

| 度量 | 公式直觉 | 含义 | 何时用 |
|------|---------|------|--------|
| **余弦 Cosine** | 夹角余弦 `a·b/(|a||b|)` | 只看方向，不看长度 | 文本检索最常用 |
| **点积 Dot** | `Σ aᵢbᵢ` | 方向+长度都有影响 | 归一化后 ≈ 余弦；向量库常用 |
| **欧氏 Euclidean** | `‖a-b‖` | 空间里的直线距离 | 几何距离场景 |

三者关系：若向量都已归一化（长度=1），则 `dot == cosine`，且 `euclidean² = 2 - 2·cosine`。所以很多向量库直接用点积近似余弦。

In [ ]:
# 用 numpy 手写三个度量，并验证“归一化后三者等价”
def cosine(a, b):
    return float(a @ b / ((np.linalg.norm(a) * np.linalg.norm(b)) + 1e-10))
def euclidean(a, b):
    return float(np.linalg.norm(a - b))

if API_KEY and '你的' not in API_KEY:
    a = v[0]; b = v[1]; c = v[2]
    print('相关两句话  余弦:', round(cosine(a, b), 3), '| 欧氏:', round(euclidean(a, b), 3))
    print('无关两句话  余弦:', round(cosine(a, c), 3), '| 欧氏:', round(euclidean(a, c), 3))
    print('=> 语义相近: 余弦高、欧氏小；语义无关则反之。')
else:
    # 无 key 时的本地演示
    a = np.array([1.0, 0.0]); b = np.array([0.9, 0.1]); c = np.array([0.0, 1.0])
    print('示例(二维): a≈b 余弦', round(cosine(a, b), 2), 'vs a≈c 余弦', round(cosine(a, c), 2))

## 3. 一次“最简检索”验证性质

把问题向量与若干候选向量比相似度、取 Top，本质就是 dense retrieval（第 13 课）的最小原型。



In [ ]:
# 知识点·真调说明：最简检索 Top-K —— 把问题向量与候选向量比相似度、取 Top，跑通 dense retrieval 的最小原型
# 复用本课上面定义好的 embed()（真向量）与 cosine()：看 Top1 是否命中“真相关”的那条。
import os
import numpy as np

_HAS = bool(os.getenv('DASHSCOPE_API_KEY', '').strip()) and '你的' not in os.getenv('DASHSCOPE_API_KEY', '')
if _HAS:
    query = '客服机器人怎么收费'
    docs = ['价格套餐有哪些档位？', '今天食堂做了红烧牛肉面', '申请年假需要提前几天']
    q = embed([query])[0]
    vd = embed(docs)
    scores = [(cosine(q, d), t) for d, t in zip(vd, docs)]
else:
    print('未配置 Key，先用一组示意向量走同一套排序逻辑（配置 Key 后自动改用真实文本向量）。')
    query, docs = '客服机器人怎么收费', ['价格套餐有哪些档位？', '今天食堂做了红烧牛肉面', '申请年假需要提前几天']
    q = np.array([1.0, 0.0])
    vd = np.array([[0.95, 0.31], [-0.20, 0.98], [0.05, 0.99]])
    scores = [(cosine(q, d), t) for d, t in zip(vd, docs)]

rank = sorted(scores, key=lambda x: x[0], reverse=True)
print('查询：', query)
for i, (s, t) in enumerate(rank, 1):
    print(f'Top{i}  相似度 {s:.3f}   {t}')
print('→ “价格套餐…”与问句“怎么收费”用词几乎不重叠却排第一，另外两句明显更低——向量按“语义远近”而非字面排序。')
print('  把这段“query 与候选逐一比相似度、取 Top”的循环交给向量库，就是第 13 课要讲的 dense retrieval。')

## 小结

- Embedding 把语义变成稠密向量坐标；
- 文本相似度常用余弦；归一化后可直接用点积；欧氏是它的几何对应；
- “归一化 + 点积”= 余弦，是向量库的标配实现。